<a href="https://colab.research.google.com/github/gibthom12-arch/PythonGameManagementSystem/blob/main/CourseWorkSub.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Game Store Management System
## What should be uploaded
FeedBackManager.txt, Gameslist.txt, UserIDManager.txt, FileOp1.py

## Load Game
This loads all the games and outputs them along with their stock. When renting or adding new games please remember to load the game stock

## Add New Games
This allows you to enter a new game and its stock for customers to rent.

## Rent Games
This is where users can rent any game currently in stock which will then be saved in the UserIDManager alongside the game and the date they rented the product. To add to this users can also see the feedback of each game by clicking the Feedback button on the desired game. Users may also search their desired game using the text box to quickly find any game in the catalog

## FeedBack
Feedback is a series of functions in my fileOp1 program which enters the data along with game name into the FeedBackManager.txt file which can then be read for each specific game

## Return Games
Here all the unreturned games a user has rented will appear at which they will be given a choice to return it. Once a game is returned it gets added back to the stock to allow a new user to rent it and the return date is saved in the UserIDManager. Finally after returning the user may enter some feedback for the game which can be seen by other users.

## Game Statistics
Game statistics shows a pie chart of the most popular games based on the amount of times each game has been rented. And as a result showing games which may be performing worse



In [ ]:
from google.colab import files
uploaded = files.upload()


Saving FeedBackManager.txt to FeedBackManager.txt
Saving FileOp1.py to FileOp1.py
Saving Gameslist.txt to Gameslist.txt
Saving Subscription_Info.txt to Subscription_Info.txt
Saving subscriptionManager.pyc to subscriptionManager.pyc
Saving UserIDManager.txt to UserIDManager.txt


In [ ]:
from os import name
from re import M
from logging import PlaceHolder
from typing import List
import ipywidgets as widgets
import FileOp1 as fo
from ipywidgets import Layout, Image
from IPython.display import clear_output
import requests
from io import BytesIO
import matplotlib.pyplot as plt
%matplotlib inline
import numpy as np
import subscriptionManager as smSL

#These are all the global variables defined for later in the Program.
PressedShowButton = False
Subscriptions = None
CurrentUserID = None
CurrentGame = None
ListOGames = []
ListOImages = ["https://toppng.com/uploads/preview/ta-5-funny-moments-grand-theft-auto-gta-v-five-5-ps4-game-1156352941445ainuqzln.png", "https://cdn.prod.website-files.com/64ea57571d50b02423c4505d/64fa5f649846f59218adca46_minecraft%20logo%20png.png", "https://toppng.com/uploads/preview/ark-survival-evolved-logo-transparent-11562971982a3jbeu6wo4.png", "https://m.media-amazon.com/images/I/91CbAezBIxL.png", "https://images-wixmp-ed30a86b8c4ca887773594c2.wixmp.com/i/3367fa99-0ba6-454c-b947-683f1a9f896d/ddwh3ph-92b34e9e-b5fb-4507-a8e1-1253c19b21b9.png", "https://image.api.playstation.com/vulcan/ap/rnd/202108/0410/2odx6gpsgR6qQ16YZ7YkEt2B.png", "https://sm.pcmag.com/pcmag_uk/news/h/halo-comba/halo-combat-evolved-anniversary_av63.png", "https://images.seeklogo.com/logo-png/33/1/fortnite-logo-png_seeklogo-330839.png", "https://static.wikia.nocookie.net/howtotrainyourdragon/images/c/c9/Rise_of_Berk_logo.png/revision/latest?cb=20161229112019", "https://www.pngfind.com/pngs/m/527-5278868_kisspng-chess-piece-chessboard-pawn-vector-hand-painted.png"]

#These are the text boxes and inputs defined earlier which are then displayed when needed
UserIdInp = widgets.Text(placeholder = "Please enter your UserId", layout = Layout(border = "2px solid black", width = "30%"))
Rent = widgets.Label(value = "", layout = Layout(border = "", width = ""))
SearchInp = widgets.Text(placeholder = "Please enter a game to search", layout = Layout(border = "2px solid black", width = "30%"))
NewGamesInp = widgets.Text(placeholder = "Please enter a game and its stock", layout = Layout(border = "2px solid black", width = "30%"))
NewGamesInfo = widgets.Label(value = "", layout = Layout(border = "", width = ""))
AddNewGamesInfo = widgets.Label(value = "", layout = Layout(border = "", width = ""))
MoreInfo = widgets.Label(value = "", layout = Layout(border = "", width = ""))
FeedBackInp = widgets.Text(placeholder = "Please enter FeedBack", layout = Layout(border = "2px solid black", width = "20%"))

#This functions checks the UserID using the SubscriptionManager.pyc file and validates it. If it is valid it may shown the rest of the options
def UserIDCheck(change):
    global Subscriptions
    global CurrentUserID
    global PressedShowButton
    UserID = change.value.strip().lower()
    if not UserID:
      print("Please enter a UserId")
      return
    if not UserID.isalpha() or len(UserID) != 4:
      print("Invalid format")
      return
    try:
      Subscriptions = smSL.load_subscriptions("Subscription_Info.txt")
      if not smSL.check_subscription(UserID, Subscriptions):
        print("User has no active subscription")
        return
      CurrentUserID = UserID
      ShowButtons(PressedShowButton)
      PressedShowButton = True
      return
    except KeyError:
      print("UserID not found in subscription database")
      return
    except FileNotFoundError:
      print("Subscription database not found")
      return

#This functions stores the list of games from the GamesList.txt file in the global variable ListOGames
def GListbutton_click(btn):
  global ListOGames
  with output:
    clear_output()
    Rent.value = "Returning list of Games"
    display(Rent)
    templistOGames = fo.AvailableGame()
    ListOGames = templistOGames
    print(ListOGames)

#This displays the GUI for when a Add New Games button is clicked
def NewGames_click(btn):
  with output:
    clear_output()
    NewGamesInfo.value = "Please enter a game and its stock and click enter"
    AddNewGamesInfo.value = "Example: Minecraft,9"
    MoreInfo.value = "Please load game stock after"
    Info = widgets.VBox([NewGamesInfo, AddNewGamesInfo, MoreInfo])
    display(NewGamesInp, Info)
    NewGamesInp.value = ""
    NewGamesInp.on_submit(AddNewGameHandler)
    return

#This function first checks the users subscription limit and if permissable writes the rented game into the UserIDManager
def Rent_click(game_name):
  with output:
    clear_output()
    GamesRented = fo.OwnedGames(CurrentUserID, False)
    rentalLimit = smSL.get_rental_limit(Subscriptions[CurrentUserID]["SubscriptionType"])
    if len(GamesRented) < int(rentalLimit):
      fo.GameRent(CurrentUserID, game_name)
    else:
      print("Rental limit reached")
    return

#Due to the addition of the add games function I found it most efficient to dynamically make cards for each game so that when a new game is added it may also be rented
#Thus this program goes through each game in ListOGames and makes a card for it one by one
def Rental_Button_Production(btn):
  with output:
    clear_output()
    global ListOGames

    if not ListOGames:
      print("Please load the game stock")
      return

    GameCards = []
    for game in ListOGames:
      game_name = game[0]
      game_stock = game[1]

      name_label = widgets.Label(value=f"Game: {game_name}")
      stock_label = widgets.Label(value=f"Stock: {game_stock}")
      rent_button = widgets.Button(description=f"Rent: {game_name}")
      rent_button.on_click(lambda b, name = game_name: Rent_click(name))
      Read_FeedBack_btn = widgets.Button(description=f"FeedBack: {game_name}")
      Read_FeedBack_btn.on_click(lambda b, name = game_name: FeedBack_Click(name))

      if ListOGames.index(game) < 10:
        ImageURL = ListOImages[ListOGames.index(game)]
        response = requests.get(ImageURL)
        if response.status_code == 200:
          image_data = BytesIO(response.content).getvalue()
          gameimage = widgets.Image(value=image_data, format='png', width=100, height=100)
        else:
          gameimage = widgets.Image(value=b'', format='png', width=100, height=100)
      else:
          gameimage = widgets.Image(value=b'', format='png', width=100, height=100)

      GameCardsContent = [gameimage, name_label, stock_label, rent_button, Read_FeedBack_btn]
      GameCard = widgets.VBox(GameCardsContent, layout = Layout(border = "2px solid lightblue", padding = "10px", margin = "5px", width = "200px"))
      GameCards.append(GameCard)
    display(SearchInp)
    display(widgets.VBox(GameCards, layout = Layout(flex_flow = "row wrap", justify_content = "flex-start")))

#This looks searches for the specific feedback of a game and displays it when found
def FeedBack_Click(name):
  with output:
    clear_output()
    AllFeedback = fo.FeedBackSearcher()
    for Lines in AllFeedback:
      if name == Lines[0]:
        print("FeedBack:", Lines[1])
    return

#This displays the GUI for games that are to be returned
#Similarly to the Rental button production this dynamically creates cards for when a new game is rented and displays it
def ReturnGame(btn):
  with output:
    clear_output()
    global GamesRented
    Returning = False
    RentedGames = fo.OwnedGames(CurrentUserID, False)
    ReturnCards = []
    for game in RentedGames:
      name_label = widgets.Label(value=f"Game: {game}")
      return_button = widgets.Button(description=f"Return: {game}")
      return_button.on_click(lambda b, name = game: ReturnGame_Click(name))

      ReturnCardsContent = [name_label, return_button]
      ReturnCard = widgets.VBox(ReturnCardsContent, layout = Layout(border = "2px solid lightblue", padding = "10px", margin = "5px", width = "200px"))
      ReturnCards.append(ReturnCard)
    display(widgets.VBox(ReturnCards, layout = Layout(flex_flow = "row wrap", justify_content = "flex-start")))
    if not RentedGames:
      print("No games to return")
    return

#This function takes in the name of the game returned from the ReturnGame function and passes it into ReturnGame function with the userID in fo to be written to be returned
#It also displays a text box where a user can enter their feedback which is then written to the feedbackManager.txt file
def ReturnGame_Click(name):
  global CurrentGame
  with output:
    clear_output()
    fo.ReturnGame(CurrentUserID, name)
    print(f"{name} has been returned")
    CurrentGame = name
    FeedBackInp.value = ""
    display(FeedBackInp)
    FeedBackInp.on_submit(FeedBackSubmitHandler)
    print("Please click enter after inputting feedback")
    return

#This is such that when the enter key is pressed the game and stock is submitted
def AddNewGameHandler(change):
  with output:
    clear_output()
    NewGame = change.value.strip()
    if not NewGame:
      print("Please enter a game")
      return
    SplitNewGame = NewGame.split(",")
    if len(SplitNewGame) != 2:
      print("Please enter a game and stock in format: Game_Name,Stock")
      return
    stock = SplitNewGame[1].strip()
    if not stock.isdigit():
      print("Please enter a valid stock")
      return
    if int(stock) < 0:
      print("Stock cannot be negative")
      return
    fo.AddNewGames(NewGame)

#This function works so that when the enter key is pressed the feedback entered may be submitted and passed to the FeedBackWriter to be written to the FeedbackManager.txt file
def FeedBackSubmitHandler(change):
  global CurrentGame
  with output:
    clear_output()
    FeedBack = change.value.strip()
    if not FeedBack:
      print("Please enter feedback")
      return
    fo.FeedBackWriter(CurrentGame, FeedBack)

#This function activates when the GameStats button is clicked and displays the rental statistics of each games using a pie chart
def GameStats_click(btn):
    with output:
      clear_output()
      global ListOGames
      AllRentedGames = fo.OwnedGames(CurrentUserID, True)
      game_count = {}
      if not AllRentedGames:
        print("No rental data available")
        return
      if not ListOGames:
        print("Please load the game stock")
        return
      CleanedRentedGames = [g.strip() for g in AllRentedGames]

      for game in ListOGames:
        count = 0
        GameName = game[0].strip()
        for rented_game in CleanedRentedGames:
          if GameName == rented_game:
            count = count + 1
          if count > 0:
            game_count[game[0]] = count

      PlotGames = []
      PlotStock = []

      for game, count in game_count.items():
        PlotGames.append(game)
        PlotStock.append(count)

      plt.figure()
      plt.pie(PlotStock, labels=PlotGames, autopct='%1.1f%%')
      plt.title('Most Rented Games')
      plt.axis('equal')
      plt.show()

      LeastPopularGames = []
      for game in ListOGames:
        game_name = game[0].strip()
        if game_name not in game_count:
          LeastPopularGames.append(game)

      print("Least Popular Games:")
      if not LeastPopularGames:
        print("All games rented atleast once")
        return
      else:
        for unpopulargames in LeastPopularGames:
          print("-", unpopulargames)

#This function works such that when searching for a game upon clicking enter it displays the specific game you are searching for if it exists
def SearchHandler(change):
  with output:
    clear_output()
    global ListOGames

    if not ListOGames:
      print("Please load the game stock")
      return

    result = change.value.strip()
    LowerResult = result.lower()
    if not LowerResult:
      print("Please enter a search")
      return
    change.value = ""
    FilteredGames = []
    for game in ListOGames:
      game_name = game[0].strip()
      lowergame_name = game_name.lower()
      if LowerResult in lowergame_name:
        FilteredGames.append(game)

    SearchCards = []
    for game in FilteredGames:
      name_label = widgets.Label(value=f"Game: {game[0]}")
      stock_label = widgets.Label(value=f"Stock: {game[1]}")
      rent_button = widgets.Button(description=f"Rent: {game[0]}")
      rent_button.on_click(lambda b, name = game[0]: Rent_click(name))

      SearchCardsContent = [name_label, stock_label, rent_button]
      SearchCard = widgets.VBox(SearchCardsContent, layout = Layout(border = "2px solid lightblue", padding = "10px", margin = "5px", width = "200px"))
      SearchCards.append(SearchCard)
    display(widgets.VBox(SearchCards, layout = Layout(flex_flow = "row wrap", justify_content = "flex-start")))

output = widgets.Output()

#All the buttons and their on click functions
GamesStock = widgets.Button(description = "Load Game Stock")
GamesStock.on_click(GListbutton_click)
NewGames = widgets.Button(description = "Add New Game")
NewGames.on_click(NewGames_click)
GamesRent = widgets.Button(description = "Rent Game")
GamesRent.on_click(Rental_Button_Production)
GamesReturn = widgets.Button(description = "Return Game")
GamesReturn.on_click(ReturnGame)
GameStats = widgets.Button(description = "Game Statistics")
GameStats.on_click(GameStats_click)
SearchInp.on_submit(SearchHandler)
UserIdInp.on_submit(UserIDCheck)
buttons = widgets.HBox([])

#This function waits till activation from UserIDInp value being returned as Valid
#After this function activates once and is prevented from reactivating due to the Boolean Pressed being permanently changed to True in the UserIDCheck after activating once
def ShowButtons(Pressed):
  if Pressed == False:
    buttons = widgets.HBox([GamesStock, NewGames, GamesRent, GamesReturn, GameStats])
    display(buttons, output)
  return

display(UserIdInp)


ModuleNotFoundError: No module named 'FileOp1'